# LangChain in Its Modern Role
**Module 5, Topic 4 — Automation & Agentic Systems**

## The Problem

In Topic 3 we built the compliance-checker agent with hand-written code: `open().read()` for loading documents, `text.split("\n\n")` for chunking, and keyword matching for search. This works, but it is **brittle** — it misses semantic matches (e.g. "licence violation" won't match a paragraph about "non-compliance"), handles no overlap between chunks, and loses document metadata.

## The Solution

**LangChain** provides production-quality components — document loaders, text splitters, embedding-based vector stores — that drop into the same LangGraph graph without changing its structure. The graph stays the same; only the parts inside the search node improve.

In this notebook you will:
1. Load documents with `TextLoader` (metadata preserved)
2. Split them with `RecursiveCharacterTextSplitter` (overlap handled)
3. Build a ChromaDB vector store and retriever
4. Drop the retriever into the Topic 3 graph
5. Compare keyword vs. similarity search results


## Setup

In [ ]:
!pip install langgraph openai python-dotenv langchain-core langchain-community langchain-text-splitters langchain-openai chromadb -q

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)
MODEL = os.getenv("MODEL_NAME")

print(f"Client ready — model: {MODEL}")

## Step 1 — Load Documents with LangChain

In Topic 3 we used `open()`. LangChain's `TextLoader` returns `Document` objects with metadata.

In [ ]:
from langchain_community.document_loaders import TextLoader

CORPUS_DIR = os.path.join("..", "..", "..", "..", "shared", "synthetic-mce-corpus")

files = [
    "MCE-CN-2026-031_compliance_notice.txt",
    "MCE-EC-2025-118_enforcement_case_file.txt",
    "NOC-INC-2026-0417_noc_incident_report.txt",
]

all_docs = []
for filename in files:
    path = os.path.join(CORPUS_DIR, filename)
    loader = TextLoader(path, encoding="utf-8")
    docs = loader.load()
    all_docs.extend(docs)

print(f"Loaded {len(all_docs)} documents")
for doc in all_docs:
    print(f"  {doc.metadata['source']}: {len(doc.page_content)} chars")

## Step 2 — Split Documents with RecursiveCharacterTextSplitter

In Topic 3 we split on `\n\n` by hand. The LangChain splitter handles overlap and edge cases.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "],
)

chunks = splitter.split_documents(all_docs)

print(f"Split into {len(chunks)} chunks")
for i, chunk in enumerate(chunks[:5]):
    source = os.path.basename(chunk.metadata['source'])
    print(f"  Chunk {i}: {len(chunk.page_content)} chars from {source}")

## Step 3 — Build a Vector Store and Retriever

Embed the chunks with OpenAI embeddings and store them in ChromaDB.  
`.as_retriever()` gives us a callable that accepts a query and returns matching documents.

In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

embeddings = OpenAIEmbeddings(
    api_key=os.getenv("EMBEDDING_API_KEY", os.getenv("OPENAI_API_KEY")),
    base_url=os.getenv("EMBEDDING_BASE_URL"),
    model=os.getenv("EMBEDDING_MODEL"),
)

vectorstore = Chroma.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

print("Vector store built, retriever ready")

## Step 4 — Test the Retriever

Compare: in Topic 3 we used keyword matching. The retriever uses similarity search.

In [ ]:
test_results = retriever.invoke("Sahel Link compliance deadline")

print(f"Found {len(test_results)} results\n")
for i, doc in enumerate(test_results):
    source = os.path.basename(doc.metadata['source'])
    print(f"Result {i+1} [{source}]:")
    print(f"  {doc.page_content[:120]}...\n")

## Step 5 — Build the Agent with the LangChain Retriever

Same graph as Topic 3 — only the search node changes.

In [ ]:
from typing import TypedDict, Annotated
from operator import add
from langgraph.graph import StateGraph, START, END


class ComplianceState(TypedDict):
    question: str
    search_results: Annotated[list, add]
    draft: str
    needs_search: bool
    approved: bool
    answer: str


def decide(state: ComplianceState) -> dict:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system",
             "content": "Decide if this compliance question requires searching "
                        "compliance documents. Reply YES or NO only."},
            {"role": "user", "content": state["question"]},
        ],
    )
    needs = response.choices[0].message.content.strip().upper().startswith("YES")
    print(f"[decide] needs_search = {needs}")
    return {"needs_search": needs}


def search(state: ComplianceState) -> dict:
    """Search using the LangChain retriever instead of keyword matching."""
    docs = retriever.invoke(state["question"])
    results = [
        {"doc_id": os.path.basename(d.metadata.get("source", "unknown")),
         "excerpt": d.page_content}
        for d in docs
    ]
    print(f"[search] found {len(results)} results (via LangChain retriever)")
    return {"search_results": results}


def draft(state: ComplianceState) -> dict:
    if state["search_results"]:
        context = "\n\n".join(
            f"[{r['doc_id']}]: {r['excerpt']}" for r in state["search_results"]
        )
    else:
        context = "(no documents found)"
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system",
             "content": "You are a compliance assistant. Answer using only the "
                        "documents below. Cite each fact with the document "
                        "reference in square brackets. If the documents do not "
                        "contain the answer, say so."},
            {"role": "user",
             "content": f"Question: {state['question']}\n\nDocuments:\n{context}"},
        ],
    )
    draft_text = response.choices[0].message.content
    print(f"[draft] wrote {len(draft_text)} chars")
    return {"draft": draft_text}


def review(state: ComplianceState) -> dict:
    print("=" * 60)
    print("DRAFT FOR REVIEW")
    print("=" * 60)
    print(state["draft"])
    print("=" * 60)
    print("[auto-approved for demo]")
    return {"approved": True}


def finalise(state: ComplianceState) -> dict:
    print("[finalise] done")
    return {"answer": state["draft"]}


def route_after_decide(state):
    return "search" if state["needs_search"] else "draft"

def route_after_review(state):
    return "finalise" if state["approved"] else "search"


print("All nodes and routing functions defined")

In [ ]:
graph = StateGraph(ComplianceState)

graph.add_node("decide", decide)
graph.add_node("search", search)
graph.add_node("draft", draft)
graph.add_node("review", review)
graph.add_node("finalise", finalise)

graph.add_edge(START, "decide")
graph.add_conditional_edges("decide", route_after_decide,
    {"search": "search", "draft": "draft"})
graph.add_edge("search", "draft")
graph.add_edge("draft", "review")
graph.add_conditional_edges("review", route_after_review,
    {"finalise": "finalise", "search": "search"})
graph.add_edge("finalise", END)

app = graph.compile()
print("Graph compiled — same structure as Topic 3, LangChain retriever inside")

## Step 6 — Test: Compliance Question

Same question as Topic 3. Compare the retrieval quality — does the similarity search find the revision notice?

In [ ]:
result = app.invoke({
    "question": "Is Sahel Link Networks currently compliant with Licence Condition 7.2?",
    "search_results": [],
    "draft": "",
    "needs_search": False,
    "approved": False,
    "answer": "",
})

print("\nFINAL ANSWER:")
print(result["answer"])

### Check

- Does it cite the document source?
- Does it give the **revised deadline 2026-04-13**?
- Compare with the Topic 3 result — did the similarity search find more relevant paragraphs than keyword matching?

## Side-by-Side: What Changed, What Stayed

| Component | Topic 3 (hand-written) | Topic 4 (LangChain) |
|-----------|----------------------|--------------------|
| Document loading | `open().read()` | `TextLoader` |
| Splitting | `text.split("\n\n")` | `RecursiveCharacterTextSplitter` |
| Search | keyword matching | ChromaDB similarity via `.as_retriever()` |
| LLM calls | raw OpenAI client | raw OpenAI client (unchanged) |
| Graph structure | 5 nodes, 2 conditional edges | 5 nodes, 2 conditional edges (unchanged) |
| State schema | ComplianceState TypedDict | ComplianceState TypedDict (unchanged) |

The graph did not change. Only the components inside the search node changed.

## Summary

- **LangChain loaders** read documents into a standard format with metadata
- **RecursiveCharacterTextSplitter** handles overlap and edge cases better than hand-written splitting
- **`.as_retriever()`** wraps a vector store into a callable that fits inside any LangGraph node
- **The graph did not change** — LangGraph handles orchestration, LangChain provides better components
- **Use LangChain where it saves work, use the raw API where you want control**

## Exercises

### Exercise 1 — Tune chunk size
Rebuild the vector store with `chunk_size=250` and `chunk_overlap=25`. Re-run the compliance question. Does smaller chunk size improve or hurt the answer? Why?

### Exercise 2 — Change retriever `k`
Create a second retriever with `search_kwargs={"k": 8}` (double the default). Run the enforcement question from Topic 3 (*"What was the outcome of the Delta Coast Broadband enforcement case?"*). Does returning more chunks improve the citation quality, or add noise?

### Exercise 3 — Semantic vs. keyword side-by-side
Import the `search_corpus` keyword function from Topic 3 (copy it into a cell). For the query *"licence violation penalties"*, run both the keyword search and the LangChain retriever. Print the results side by side. Which one finds the relevant paragraphs? Why does the keyword search miss them?

**Time:** 20 minutes

In [ ]:
# Exercise 1 — your code here


In [ ]:
# Exercise 2 — your code here


In [ ]:
# Exercise 3 — your code here
